# Código para limpiar y transformar el precio OMIE

In [4]:
import pandas as pd

# =========================
# 1. CARGA del CSV
# =========================

df_price = pd.read_csv(
    "OMIEPrecio2020_2024.csv", 
    sep=";"
)

# Eliminar columnas vacias si hay
df_price = df_price.dropna(axis=1, how="all")

# =========================
# 2. QUEDARNOS SOLO CON PRECIO
# =========================

df_price = df_price[df_price["CONCEPT"] == "PRICE_SP"]

# =========================
# 3. PASAR DE ANCHO A LARGO
# =========================

# En lugar de tener las horas en columnas, tener una fila por hora
hour_cols = [col for col in df_price.columns if col.startswith("H")]

df_long = df_price.melt(
    id_vars=["DATE"],
    value_vars=hour_cols,
    var_name="hour",
    value_name="price"
)

# =========================
# 4. LIMPIEZA HORA
# =========================

# Extraer número de hora
df_long["hour"] = df_long["hour"].str.replace("H", "").astype(int)

# Tenemos H1 = 00:00-01:00 y lo lo pasamos a formato 0-23 
df_long["hour"] = df_long["hour"] - 1

# =========================
# 5. CREAR DATETIME
# =========================

df_long["datetime"] = pd.to_datetime(df_long["DATE"]) + \
                      pd.to_timedelta(df_long["hour"], unit="h")

# =========================
# 6. LIMPIEZA FINAL
# =========================

df_long = df_long.drop(columns=["DATE"])
df_long["price"] = pd.to_numeric(df_long["price"], errors="coerce")

df_long = df_long.sort_values("datetime").reset_index(drop=True)

# =========================
# RESULTADO FINAL
# =========================

print(df_long.head())
print(df_long.info())

# =========================
# Guardarlo en un CSV
# =========================

df_long.to_csv("limpiezaPrecio_Omie2020_2024.csv", sep=';', index=None)


   hour  price            datetime
0     0  41.88 2020-01-01 00:00:00
1     1  38.60 2020-01-01 01:00:00
2     2  36.55 2020-01-01 02:00:00
3     3  32.32 2020-01-01 03:00:00
4     4  30.85 2020-01-01 04:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45675 entries, 0 to 45674
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   hour      45675 non-null  int64         
 1   price     43848 non-null  float64       
 2   datetime  45675 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.0 MB
None
